In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
import  pyspark.sql.functions as fun
import numpy as np
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder.getOrCreate()

In [3]:
sc = spark.sparkContext

Create an RDD from a list of numbers (1,50) using numpy methods

In [4]:
nums = np.arange(1, 51)
rdd = sc.parallelize(nums)
print(rdd.collect())

[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50)]


Find the sum, average, maximum, minimum, and count

In [5]:
#sum
total_sum = rdd.sum()
print("Sum:", total_sum)

Sum: 1275


In [6]:
#avg
count = rdd.count()
avg = total_sum / count
print("Average:", avg)

Average: 25.5


In [7]:
#Count
count = rdd.count()
print("Count:", count)

Count: 50


In [8]:
#Min
minimum = rdd.min()
print("Minimum:", minimum)

Minimum: 1


In [9]:
#Max
maximum = rdd.max()
print("Maximum:", maximum)

Maximum: 50


Count how many numbers are even vs. odd.

In [10]:
even_count = rdd.filter(lambda x: x % 2 == 0).count()
print("Even numbers count:", even_count)

odd_count = rdd.filter(lambda x: x % 2 != 0).count()
print("Odd numbers count:", odd_count)

Even numbers count: 25
Odd numbers count: 25


You have the following data of people info ('Name', 'Age'), answer the following questions

In [11]:
people_data = [("Nada ", 25), ("Mona", 30), ("Ahmed", 35), ("Khaled", 40),("Ahmed", 35), ('Nada ', 25)]
rdd_people = sc.parallelize(people_data)
rdd_people.collect()

[('Nada ', 25),
 ('Mona', 30),
 ('Ahmed', 35),
 ('Khaled', 40),
 ('Ahmed', 35),
 ('Nada ', 25)]

Find the oldest person

In [12]:
oldest_person = rdd_people.max(key=lambda x: x[1])
print("Oldest person:", oldest_person)

Oldest person: ('Khaled', 40)


Compute the average age

In [13]:
total_age = rdd_people.map(lambda x: x[1]).sum()
count = rdd_people.count()

avg_age = total_age / count
print("Average age:", avg_age)

Average age: 31.666666666666668


Group all the names by their age

In [14]:
grouped = rdd_people.groupByKey().mapValues(list)

for age, names in grouped.collect():
    print(f"Age {age}: {names}")

Age Ahmed: [35, 35]
Age Nada : [25, 25]
Age Mona: [30]
Age Khaled: [40]


Take the following text and put it in a text file named russia.txt and load it into rdd

"Russia is the largest country in the world by land area
Moscow is the capital city of Russia
The Russian language is one of the most widely spoken languages in the world
Russia is known for its rich history and culture
The Trans-Siberian Railway is the longest railway line in the world
Russia has a strong tradition in literature, music and ballet
The country is famous for its cold winters and vast landscapes
Russia is a major player in global energy production
"

In [18]:
from google.colab import files
uploaded = files.upload()

Saving russia.txt to russia.txt


In [20]:
rdd_russia = sc.textFile("/content/russia.txt")
rdd_russia.collect()

['"Russia is the largest country in the world by land area',
 'Moscow is the capital city of Russia',
 'The Russian language is one of the most widely spoken languages in the world',
 'Russia is known for its rich history and culture',
 'The Trans-Siberian Railway is the longest railway line in the world',
 'Russia has a strong tradition in literature, music and ballet',
 'The country is famous for its cold winters and vast landscapes',
 'Russia is a major player in global energy production',
 '"']

Count the total number of lines.

In [21]:
line_count = rdd_russia.count()
print("Total number of lines:", line_count)

Total number of lines: 9


Count how many lines contain the word "Russia"

In [22]:
russia_lines_count = rdd_russia.filter(lambda line: "Russia" in line).count()
print("Number of lines containing 'Russia':", russia_lines_count)

Number of lines containing 'Russia': 6


Find the most 5 frequent word in the file.

In [23]:
words = rdd_russia.flatMap(lambda line: line.split())
word_pairs = words.map(lambda word: (word, 1))
word_counts = word_pairs.reduceByKey(lambda a, b: a + b)
sorted_words = word_counts.sortBy(lambda x: x[1], ascending=False)
top_5_words = sorted_words.take(5)
print("Top 5 frequent words:", top_5_words)

Top 5 frequent words: [('is', 7), ('the', 7), ('in', 5), ('Russia', 4), ('world', 3)]


Tokenize words

In [24]:
tokens = rdd_russia.flatMap(lambda line: line.split())
print(tokens.take(20))

['"Russia', 'is', 'the', 'largest', 'country', 'in', 'the', 'world', 'by', 'land', 'area', 'Moscow', 'is', 'the', 'capital', 'city', 'of', 'Russia', 'The', 'Russian']


Remove stopwords (a, the, is, to, in, of).

In [25]:
import re

stopwords = {"a", "the", "is", "to", "in", "of"}
tokens = rdd_russia.flatMap(
    lambda line: re.findall(r"\b\w+\b", line.lower())
)
filtered_tokens = tokens.filter(lambda word: word not in stopwords)
print(filtered_tokens.take(20))


['russia', 'largest', 'country', 'world', 'by', 'land', 'area', 'moscow', 'capital', 'city', 'russia', 'russian', 'language', 'one', 'most', 'widely', 'spoken', 'languages', 'world', 'russia']


Count the frequency of each word

In [26]:
stopwords = {"a", "the", "is", "to", "in", "of"}
tokens = rdd_russia.flatMap(
    lambda line: re.findall(r"\b\w+\b", line.lower())
)

filtered_tokens = tokens.filter(lambda word: word not in stopwords)
word_counts = filtered_tokens.map(lambda w: (w, 1)) \
                             .reduceByKey(lambda a, b: a + b)

for word, freq in word_counts.collect():
    print(word, ":", freq)

largest : 1
country : 2
world : 3
by : 1
land : 1
area : 1
capital : 1
russian : 1
language : 1
most : 1
widely : 1
known : 1
for : 2
history : 1
and : 3
trans : 1
line : 1
music : 1
famous : 1
cold : 1
winters : 1
landscapes : 1
player : 1
energy : 1
production : 1
russia : 5
moscow : 1
city : 1
one : 1
spoken : 1
languages : 1
its : 2
rich : 1
culture : 1
siberian : 1
railway : 2
longest : 1
has : 1
strong : 1
tradition : 1
literature : 1
ballet : 1
vast : 1
major : 1
global : 1


In [27]:
schema = 'id integer, name string, age integer, salary integer'
data = [
    (1, "Ali", 25, 4000),
    (2, "Mariam", 30, 6000),
    (3, "Omar", 35, 7000),
    (4, "Sara", 28, 5000),
    (5, "Omar", 25, 6500),
    (6, "Mariam", 26, 7500)
]

df = spark.createDataFrame(data,schema)

Show schema and first 2 rows

In [28]:
df.printSchema()
df.show(2)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: integer (nullable = true)

+---+------+---+------+
| id|  name|age|salary|
+---+------+---+------+
|  1|   Ali| 25|  4000|
|  2|Mariam| 30|  6000|
+---+------+---+------+
only showing top 2 rows



Select only name and salary

In [29]:
df.select("name", "salary").show()

+------+------+
|  name|salary|
+------+------+
|   Ali|  4000|
|Mariam|  6000|
|  Omar|  7000|
|  Sara|  5000|
|  Omar|  6500|
|Mariam|  7500|
+------+------+



Find the average salary

In [30]:
df.select(fun.avg("salary").alias("average_salary")).show()

+--------------+
|average_salary|
+--------------+
|        6000.0|
+--------------+



Filter employees older than 28

In [31]:
df.filter(df.age > 28).show()

+---+------+---+------+
| id|  name|age|salary|
+---+------+---+------+
|  2|Mariam| 30|  6000|
|  3|  Omar| 35|  7000|
+---+------+---+------+



Count distinct values in the name column

In [32]:
distinct_names_count = df.select("name").distinct().count()
print("Distinct names (method 1):", distinct_names_count)

df.select(fun.countDistinct("name").alias("distinct_names")).show()

Distinct names (method 1): 4
+--------------+
|distinct_names|
+--------------+
|             4|
+--------------+



Group by a the name column and find average salary

In [33]:
df.groupBy("name") \
  .agg(fun.avg("salary").alias("average_salary")) \
  .show()

+------+--------------+
|  name|average_salary|
+------+--------------+
|  Omar|        6750.0|
|Mariam|        6750.0|
|   Ali|        4000.0|
|  Sara|        5000.0|
+------+--------------+



In [34]:
from google.colab import files
uploaded = files.upload()

Saving NullData.csv to NullData.csv


In [35]:
df1 = spark.read.csv("/content/NullData.csv", header=True, inferSchema=True) #this file in shared folder
df1.show()

+----+-----+-----+
|  Id| Name|Sales|
+----+-----+-----+
|emp1| John| NULL|
|emp2| NULL| NULL|
|emp3| NULL|345.0|
|emp4|Cindy|456.0|
+----+-----+-----+



Find the avg sales

In [36]:
df1.select(fun.avg("Sales").alias("average_sales")).show()

+-------------+
|average_sales|
+-------------+
|        400.5|
+-------------+



Replace null name with 'Unknown' and sales with the avg sales of the column

In [38]:
avg_sales = df1.select(fun.avg("Sales")).first()[0]
df_filled = df1.na.fill({"Name": "Unknown", "Sales": avg_sales})
df_filled.show()

+----+-------+-----+
|  Id|   Name|Sales|
+----+-------+-----+
|emp1|   John|400.5|
|emp2|Unknown|400.5|
|emp3|Unknown|345.0|
|emp4|  Cindy|456.0|
+----+-------+-----+

